# Pinecone Vector Search — Voyage AI Embeddings

An end-to-end notebook for creating a **Pinecone serverless index** powered by **Voyage AI** embeddings (`voyage-3-large`), instead of a local `sentence-transformers` model.

**Why Voyage?**
- State-of-the-art general-purpose retrieval quality (outperforms OpenAI `text-embedding-3-large` on public benchmarks)
- Native `input_type` distinction between `document` and `query` — prepends task-specific prompts server-side for better retrieval
- Matryoshka Representation Learning: one model, multiple output dimensions (256 / 512 / 1024 / 2048)
- Quantization support (`float`, `int8`, `uint8`, `binary`, `ubinary`) — directly reduces vectorDB storage cost
- 32K-token context length

**What this notebook does**
1. Installs `pinecone` + `voyageai`
2. Prompts for both API keys at runtime (never stored in code)
3. Creates a serverless Pinecone index sized to the Voyage output dimension
4. Embeds sample documents with `input_type="document"`
5. Upserts vectors with metadata
6. Queries with `input_type="query"` — demonstrates the document/query prompt asymmetry
7. Cleanup

## 1. Install dependencies

- `pinecone` — Pinecone SDK (v5+)
- `voyageai` — official Voyage AI Python client

In [1]:
# %pip install --quiet pinecone voyageai

## 2. Imports

In [2]:
import os
import time
from getpass import getpass

import voyageai
from pinecone import Pinecone, ServerlessSpec

## 3. Authenticate (both services)

Prefer environment variables (`PINECONE_API_KEY`, `VOYAGE_API_KEY`). If either is missing, we fall back to `getpass` so the keys never land in cell output or version control.

- Pinecone key: https://app.pinecone.io → *API Keys*
- Voyage key:   https://dash.voyageai.com → *API Keys* (free tier includes generous monthly tokens)

In [3]:
pinecone_key = os.getenv("PINECONE_API_KEY") or getpass("Enter your Pinecone API key: ")
voyage_key   = os.getenv("VOYAGEAI_API_KEY")   or getpass("Enter your Voyage AI API key: ")

pc = Pinecone(api_key=pinecone_key)
vo = voyageai.Client(api_key=voyage_key)

print("Pinecone indexes in project:", [idx["name"] for idx in pc.list_indexes()])

Pinecone indexes in project: ['sample-movies']


## 4. Configure the Voyage embedding model

Key knobs exposed by `voyage-3-large` (and `voyage-4-large`):

| Parameter | Options | Notes |
|---|---|---|
| `model` | `voyage-3-large`, `voyage-3.5`, `voyage-3.5-lite`, `voyage-code-3`, `voyage-law-2`, `voyage-finance-2`, `voyage-4-large`, `voyage-4`, `voyage-4-lite` | Swap in a domain-specific model for code / legal / finance corpora |
| `output_dimension` | `256`, `512`, `1024` (default), `2048` | Matryoshka — lower dims = smaller vectorDB footprint, minor recall tradeoff |
| `output_dtype` | `float`, `int8`, `uint8`, `binary`, `ubinary` | Quantization — binary gives 32× storage reduction with surprisingly small quality hit |
| `input_type` | `document`, `query`, `None` | **Critical**: prepends task-specific prompts. Use `document` at ingest, `query` at search time |

For a demo, we'll stick with `float` + `1024` so results are easy to inspect. In production you'd revisit these per cost/quality targets.

In [4]:
VOYAGE_MODEL      = "voyage-code-3"   # or: "voyage-4-large", "voyage-3.5", "voyage-code-3", ...
OUTPUT_DIMENSION  = 1024               # 256 | 512 | 1024 | 2048
OUTPUT_DTYPE      = "float"            # "float" | "int8" | "uint8" | "binary" | "ubinary"

# Pinecone index must match the Voyage output dimension exactly
INDEX_NAME        = "sample-vector-index-voyage"
METRIC            = "cosine"           # Voyage embeddings are not L2-normalized by default; cosine handles this

## 5. Create a serverless Pinecone index

Idempotent: if the index already exists we reuse it. The index `dimension` is pinned to `OUTPUT_DIMENSION` above — if you change the Voyage dimension, you'll need to recreate the index.

In [5]:
if INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=OUTPUT_DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
    print(f"Created index '{INDEX_NAME}' with dimension {OUTPUT_DIMENSION}.")
else:
    existing_dim = pc.describe_index(INDEX_NAME).dimension
    if existing_dim != OUTPUT_DIMENSION:
        raise ValueError(
            f"Index '{INDEX_NAME}' exists with dimension {existing_dim}, "
            f"but you requested {OUTPUT_DIMENSION}. Delete it or pick a different name."
        )
    print(f"Index '{INDEX_NAME}' already exists — reusing it.")

index = pc.Index(INDEX_NAME)
index.describe_index_stats()

Created index 'sample-vector-index-voyage' with dimension 1024.


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 06:19:41 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '45',
                                    'x-pinecone-request-latency-ms': '44',
                                    'x-pinecone-response-duration-ms': '46'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

## 6. Sample corpus

Same 10 documents as the MiniLM version, across three topical clusters (data platforms, ML infra, sports), so you can compare retrieval quality side-by-side if you run both notebooks.

In [6]:
documents = [
    {"id": "doc-01", "text": "Databricks Unity Catalog provides centralized governance for data and AI assets.",         "category": "data-platform"},
    {"id": "doc-02", "text": "Snowflake Horizon offers a unified governance layer across databases and applications.",  "category": "data-platform"},
    {"id": "doc-03", "text": "Delta Lake brings ACID transactions and time travel to cloud object storage.",            "category": "data-platform"},
    {"id": "doc-04", "text": "Apache Iceberg is an open table format designed for huge analytic datasets.",             "category": "data-platform"},
    {"id": "doc-05", "text": "Vector databases store high-dimensional embeddings for semantic search.",                 "category": "ml-infra"},
    {"id": "doc-06", "text": "Retrieval-augmented generation grounds language model responses in external documents.", "category": "ml-infra"},
    {"id": "doc-07", "text": "Fine-tuning adapts a pretrained foundation model to a specific downstream task.",         "category": "ml-infra"},
    {"id": "doc-08", "text": "Liverpool won the Premier League after a dominant second-half run.",                      "category": "sports"},
    {"id": "doc-09", "text": "The cricket match ended in a draw after rain interrupted play on day four.",              "category": "sports"},
    {"id": "doc-10", "text": "Formula 1 teams unveiled new aerodynamic packages for the upcoming season.",              "category": "sports"},
]

print(f"Prepared {len(documents)} documents.")

Prepared 10 documents.


## 7. Embed documents with `input_type="document"`

Voyage transparently prepends `"Represent the document for retrieval: "` to each input before vectorizing. This is a meaningful quality lift over a generic embedding call — don't skip it.

**Rate-limit note**: Voyage caps a single `.embed()` call at 1,000 texts and a total-token budget per model (120K tokens for `voyage-3-large`). For larger corpora, batch client-side.

In [7]:
texts = [d["text"] for d in documents]

doc_result = vo.embed(
    texts=texts,
    model=VOYAGE_MODEL,
    input_type="document",
    output_dimension=OUTPUT_DIMENSION,
    output_dtype=OUTPUT_DTYPE,
)

doc_embeddings = doc_result.embeddings
print(f"Embedded {len(doc_embeddings)} documents.")
print(f"Vector dimension: {len(doc_embeddings[0])}")
print(f"Total tokens billed: {doc_result.total_tokens}")

Embedded 10 documents.
Vector dimension: 1024
Total tokens billed: 125


## 8. Upsert into Pinecone

Metadata payload carries the raw text + category so we can both inspect matches and run filtered queries later.

In [8]:
vectors = [
    {
        "id": doc["id"],
        "values": emb,
        "metadata": {"text": doc["text"], "category": doc["category"]},
    }
    for doc, emb in zip(documents, doc_embeddings)
]

# Pinecone recommends batches of ~100 for upserts
BATCH_SIZE = 100
for i in range(0, len(vectors), BATCH_SIZE):
    index.upsert(vectors=vectors[i : i + BATCH_SIZE])

print(f"Upserted {len(vectors)} vectors.")
time.sleep(2)  # upserts are async server-side
index.describe_index_stats()

Upserted 10 vectors.


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '184',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 06:19:48 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}

## 9. Query with `input_type="query"`

This is the asymmetry that matters. Voyage prepends a different prompt at query time (`"Represent the query for retrieving supporting documents: "`). Queries and documents end up in the same vector space but are instructed slightly differently — aligning them with how the model was trained for retrieval.

Rule of thumb: **always pass `input_type="query"` for search-time embeddings, `input_type="document"` for ingest.**

In [9]:
def embed_query(text: str) -> list[float]:
    result = vo.embed(
        texts=[text],
        model=VOYAGE_MODEL,
        input_type="query",
        output_dimension=OUTPUT_DIMENSION,
        output_dtype=OUTPUT_DTYPE,
    )
    return result.embeddings[0]


def pretty_print(query: str, matches):
    print(f"Query: {query}\n")
    for m in matches:
        print(f"  [{m['score']:.4f}]  {m['id']}  ({m['metadata']['category']})")
        print(f"           {m['metadata']['text']}\n")

### 9a. Semantic search

In [10]:
query_text = "How do I govern data assets on a lakehouse?"
results = index.query(
    vector=embed_query(query_text),
    top_k=3,
    include_metadata=True,
)
pretty_print(query_text, results["matches"])

Query: How do I govern data assets on a lakehouse?

  [0.6271]  doc-01  (data-platform)
           Databricks Unity Catalog provides centralized governance for data and AI assets.

  [0.5824]  doc-02  (data-platform)
           Snowflake Horizon offers a unified governance layer across databases and applications.

  [0.5071]  doc-03  (data-platform)
           Delta Lake brings ACID transactions and time travel to cloud object storage.



### 9b. Semantic search with metadata filter

Same query pattern, but restricted to `ml-infra`. Pinecone applies the filter server-side before the ANN search — no wasted compute on out-of-scope candidates.

In [11]:
query_text = "What technique grounds LLMs in external knowledge?"
results = index.query(
    vector=embed_query(query_text),
    top_k=3,
    include_metadata=True,
    filter={"category": {"$eq": "ml-infra"}},
)
pretty_print(query_text, results["matches"])

Query: What technique grounds LLMs in external knowledge?

  [0.6298]  doc-06  (ml-infra)
           Retrieval-augmented generation grounds language model responses in external documents.

  [0.4998]  doc-07  (ml-infra)
           Fine-tuning adapts a pretrained foundation model to a specific downstream task.

  [0.4415]  doc-05  (ml-infra)
           Vector databases store high-dimensional embeddings for semantic search.



### 9c. Sanity check: does `input_type` actually matter?

Let's embed the same query string twice — once as `document`, once as `query` — and compare cosine similarity against the doc-side embeddings. You should see the `query`-typed version produce higher scores against the correct match.

In [12]:
import math
import time

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb)


def embed_with_retry(texts, input_type, max_attempts=3):
    """Voyage free tier = 3 RPM / 10K TPM. Back off and retry on rate-limit errors."""
    for attempt in range(max_attempts):
        try:
            return vo.embed(
                texts=texts,
                model=VOYAGE_MODEL,
                input_type=input_type,
                output_dimension=OUTPUT_DIMENSION,
                output_dtype=OUTPUT_DTYPE,
            ).embeddings[0]
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ("rate", "limit", "429", "payment")):
                wait = 30 * (attempt + 1)  # 30s, 60s, 90s
                print(f"  Rate-limited — sleeping {wait}s before retry {attempt + 1}/{max_attempts}...")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("Max retries exceeded. Add a payment method at https://dashboard.voyageai.com to lift the free-tier 3 RPM cap.")


sample_query = "What technique grounds LLMs in external knowledge?"
target_doc   = documents[5]["text"]   # doc-06 — the RAG sentence
target_emb   = doc_embeddings[5]

# Free tier is 3 RPM. Space calls ~22s apart to stay inside the rolling window.
# Remove the sleeps once you're on the standard tier.
print("Embedding three variants (free tier = ~1 min total)...")
as_query = embed_with_retry([sample_query], input_type="query")
time.sleep(22)
as_doc   = embed_with_retry([sample_query], input_type="document")
time.sleep(22)
as_none  = embed_with_retry([sample_query], input_type=None)

print(f"\nTarget document: {target_doc}\n")
print(f"Cosine sim (input_type='query'):    {cosine(as_query, target_emb):.4f}   ← recommended for retrieval")
print(f"Cosine sim (input_type='document'): {cosine(as_doc,   target_emb):.4f}")
print(f"Cosine sim (input_type=None):       {cosine(as_none,  target_emb):.4f}   ← no prompt prepended")

Embedding three variants (free tier = ~1 min total)...
  Rate-limited — sleeping 30s before retry 1/3...
  Rate-limited — sleeping 60s before retry 2/3...

Target document: Retrieval-augmented generation grounds language model responses in external documents.

Cosine sim (input_type='query'):    0.6304   ← recommended for retrieval
Cosine sim (input_type='document'): 0.8307
Cosine sim (input_type=None):       0.6738   ← no prompt prepended


## 10. Fetch and delete individual vectors

In [13]:
fetched = index.fetch(ids=["doc-01", "doc-05"])
for vec_id, vec in fetched.vectors.items():
    print(f"{vec_id}: {vec.metadata['text']}")

doc-05: Vector databases store high-dimensional embeddings for semantic search.
doc-01: Databricks Unity Catalog provides centralized governance for data and AI assets.


In [14]:
index.delete(ids=["doc-10"])
time.sleep(1)
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '182',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 19 Apr 2026 06:22:08 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '37',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '38'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 9}},
 'storageFullness': 0.0,
 'total_vector_count': 9,
 'vector_type': 'dense'}

## 11. Cleanup (optional)

Serverless indexes bill by storage and read/write units. Delete when done.

> Uncomment only when you actually want to tear it down.

In [15]:
# pc.delete_index(INDEX_NAME)
# print(f"Deleted index '{INDEX_NAME}'.")

## Appendix — production considerations

- **Batching**: For ingestion at scale, batch into ~100 texts per Voyage call to stay under the 120K-token-per-request budget for `voyage-3-large`. Parallelize with an async client if throughput matters.
- **Quantization**: Switching `output_dtype` to `int8` roughly quarters storage; `binary` cuts it 32× at the cost of needing a rescoring pass for top results. Only supported on `voyage-3-large`, `voyage-3.5`, `voyage-3.5-lite`, and `voyage-code-3`.
- **Matryoshka**: Dropping `output_dimension` to 512 or 256 is the cheapest lever before touching dtype — quality degrades gracefully because the model was trained for it.
- **Domain-specific models**: For code retrieval use `voyage-code-3`; legal → `voyage-law-2`; finance → `voyage-finance-2`. Each is trained on the relevant domain corpus and outperforms the general-purpose model on that vertical.
- **Rerankers**: Voyage also ships rerankers (`rerank-2.5`, `rerank-2.5-lite`). Common pattern: retrieve top-50 from Pinecone, rerank to top-5 with `voyage.rerank(...)`. Usually the highest-ROI quality upgrade you can bolt on to a RAG pipeline.